# M4b — Multilabel por técnica con XGBoost (un modelo por etiqueta)

> **El problema, en palabras simples.**
>
> El cuaderno M4 probó un modelo lineal para predecir técnicas ATT&CK por evento y chocó con un techo: T1041/T1567 no se separan (F1 0.10) y la cola queda en cero. La pregunta honesta que sigue es si el techo era del modelo lineal o de la tarea.
>
> Este cuaderno sube la capacidad al máximo razonable sin cambiar nada más: **un XGBoost binario por técnica**, mismo X (TF-IDF 5k + tabular P1–P3), mismos 6 labels, mismo corte temporal, `scale_pos_weight` por etiqueta para la cola. Si ni así se separan las cabezas, el techo es de la tarea, no del aprendiz.

> **Síntesis ejecutiva.**
>
> Misma población threat y protocolo que M4 (6 técnicas, split temporal 80/20, comparación contra M4-LogReg y oráculo-tier). Lo único que cambia es el aprendiz: 6 XGBoost binarios (300 × profundidad 6, `binary:logistic`, `scale_pos_weight = neg/pos`), etiquetas sin soporte en train quedan en ceros con nota.
>
> Resultado (`models/m4b_metrics.json`): micro-F1 0.79 (M4: 0.75), macro-F1 0.26 (M4: 0.25), hamming 0.089, subset-acc 0.73. T1190 trivial 1.00; T1041/T1567 caen a F1 0.07 (peor que el lineal); T1204 mejora a F1 0.39 cazando 10/10 con precisión 0.24; cola en cero.

| Técnica | M4-LogReg | M4b-XGB | Lectura |
|---|---|---|---|
| T1190 | 1.00 | 1.00 | Trivial en ambos |
| T1041/T1567 | F1 0.10 | F1 0.07 | Más capacidad, peor: el XGB conservador ni dispara |
| T1204 | F1 bajo | F1 0.39 (R 1.00, P 0.24) | Atrapa los 10 a base de falsos positivos |
| T1486/T1566 | 0 | 0 (skipped) | Sin soporte en live |

> **Fuentes normativas.**
>
> `notebooks/04_m4a_multilabel_logreg.ipynb` (M4, baseline lineal), EDA §§8–9, `models/m4_metrics.json` (M4) y `models/m4b_metrics.json` (este), `reports/paper.pdf` §Limitations. Binary relevance como baseline estándar del multilabel (Tsoumakas y Katakis).
>
> **Reproducción:** `uv run jupyter nbconvert --to notebook --execute notebooks/05_m4b_multilabel_xgb.ipynb --output 05_m4b_multilabel_xgb.ipynb`.

---

**Convención de lectura.**

Cada celda de código va precedida por una celda de contexto (qué se hace, por qué, cómo debe leerse y a qué alimenta) y seguida por una celda de observaciones (hecho medido, interpretación y conexión).

La sección final integra los hallazgos y fija el veredicto de despliegue.



### 1. Por qué subir la capacidad y cómo se va a evaluar

**1.1 Problema.**

M4 dejó dos lecturas posibles del techo: o el lineal era débil, o la tarea a nivel evento no es aprendible con estos features. Este cuaderno elimina la primera: XGBoost por etiqueta es el aprendiz no lineal más fuerte del proyecto (el mismo que sostiene M2/M3), con ponderación de clase por etiqueta.

**1.2 Hipótesis.**

- **H4c. Más capacidad no rescata las cabezas.** T1041/T1567 seguirán sin separarse porque en live son priors de regla derivados del tier, no firmas textuales a nivel evento. El XGB conservador disparará menos que el lineal (peor recall, F1 igual o menor).

- **H4d. La cola solo se caza con falsos positivos.** T1204 subirá en recall a costa de precisión; T1486/T1566 seguirán en cero por ausencia de soporte.

**1.3 Protocolo (idéntico a M4, cambia solo el aprendiz).**

1. **Misma población, mismo corte.** Threat live, 6 técnicas, cuantil 80 del timestamp. Comparabilidad exacta con `m4_metrics.json`.

2. **Métricas.** Micro/macro-F1, hamming, subset-acc y PR-AUC por técnica, más contraste directo contra M4 impreso en la salida.

3. **Regla de etiquetas sin soporte.** Menos de 2 positivos en train implica ceros + nota (no se entrena ruido). Se declara en `skipped`.

**1.4 Criterio de veredicto.**

Solo se despliega como segunda cabeza si alguna técnica operativa supera precisión útil para el analista. Precisiones de 0.08 mienten: un modelo que miente no se sirve, se reporta.



### 2. Carga, población threat y positivos por etiqueta en train

**2.1 Qué se hace.**

Se cargan las columnas de `signals.parquet`, se filtra threat, se parsean las 6 técnicas, se derivan `hour`, `msg_len` y `lag_h`, se binariza con `MultiLabelBinarizer` y se corta por cuantil 80, con conteo de positivos por etiqueta en train.

**2.2 Por qué.**

Idéntico punto de partida que M4: si X y el split son los mismos, toda diferencia posterior es atribuible al aprendiz. El conteo de positivos por etiqueta anticipa qué etiquetas entrenan y cuáles van a `skipped`.

**2.3 Cómo debe leerse.**

Tamaño de threat, reparto train/test y positivos por etiqueta en train (T1190 masivo, T1041/T1567 en miles, T1204 en ~10, T1486/T1566 en cero).

**2.4 Trazabilidad.**

Fija población, Y y split de las secciones 3–5. Nada del cuaderno 1 se reestima: features P1–P3 reutilizados.



In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent

COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques',
        'message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
df = pd.read_parquet(ROOT / 'data' / 'signals' / 'signals.parquet', columns=COLS)
th = df[df.label_binary != 'benign'].copy().reset_index(drop=True)
TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except Exception:
        return []
th['tech_list'] = th.attack_techniques.fillna('').map(parse)
print('threat:', th.shape, '| tecnicas:', {t: int(sum(th.tech_list.map(lambda l: t in l))) for t in TECHS})

CATS = ['message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
th['hour'] = pd.to_datetime(th['timestamp'], unit='s', utc=True).dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp'] - th['event_time']) / 3600).clip(-1e4, 1e4).fillna(-1)
for c in CATS:
    th[c] = th[c].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.8)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()}')

from sklearn.preprocessing import MultiLabelBinarizer
Y = MultiLabelBinarizer(classes=TECHS).fit_transform(th.tech_list)
print('train positives por label:', {t: int(Y[tr][:, j].sum()) for j, t in enumerate(TECHS)})

threat: (114521, 15) | tecnicas: {'T1190': 114507, 'T1041': 7742, 'T1567': 7728, 'T1204': 20, 'T1486': 0, 'T1566': 0}


train/test: 91617/22904


train positives por label: {'T1190': 91615, 'T1041': 4009, 'T1567': 4007, 'T1204': 10, 'T1486': 0, 'T1566': 0}


#### 2.1 Observaciones e insights

**Observación.**

Misma población threat que M4 (114 521), mismo corte temporal. Positivos en train concentrados en T1190/T1041/T1567; T1204 con una decena; T1486/T1566 sin soporte.

**Interpretación.**

El diseño aísla el efecto aprendiz: el experimento es M4 con el lineal sustituido por 6 XGBoost. Lo que cambie (o no) habla del techo de la tarea.

**Conexión.**

Habilita la matriz de la sección 3 y anticipa `skipped` en la 4 para la cola sin soporte.



### 3. Matriz TF-IDF 5k más tabular (idéntica a M2)

**3.1 Qué se hace.**

TF-IDF 5k ajustado solo en train más one-hot podado y numéricas, concatenación dispersa train/test.

**3.2 Por qué.**

Mismo X que M4 y M2: la comparación mide capacidad del aprendiz, no ingeniería de features. Cambiar X y aprendiz a la vez impediría atribuir el resultado.

**3.3 Cómo debe leerse.**

Dimensiones de X y confirmación de ajuste exclusivo en train (sin fuga de vocabulario futuro).

**3.4 Trazabilidad.**

Entrada de los 6 XGBoost. Reutiliza P4 sin reestimar nada del cuaderno 1.



In [2]:
# X identico a M2: TF-IDF 5k (fit train) + tabular one-hot
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS),
                           ('num', 'passthrough', ['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']])
Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
Xtr = hstack([Ttr, Atr]).tocsr(); Xte = hstack([Tte, Ate]).tocsr()
print('X:', Xtr.shape, Xte.shape)

X: (91617, 5054) (22904, 5054)


#### 3.1 Observaciones e insights

**Observación.**

Matriz dispersa con ancho dominado por el vocabulario TF-IDF, vectorizador ajustado solo en train.

**Interpretación.**

Sin fuga de vocabulario: el test futuro es verdaderamente no visto también en texto. El listón queda alto y limpio para el XGB.

**Conexión.**

Deja el terreno listo para el entrenamiento por etiqueta de la sección 4.



### 4. Seis XGBoost binarios con ponderación por etiqueta

**4.1 Qué se hace.**

Un `XGBClassifier` (`binary:logistic`, 300 × profundidad 6, `scale_pos_weight = neg/pos`) por técnica con al menos 2 positivos en train; el resto va a `skipped` y queda en ceros. Umbral 0.5 sobre la probabilidad.

**4.2 Por qué.**

Binary relevance con costo sensible por etiqueta es el estándar para colas desbalanceadas: cada cabeza aprende su propio desbalance sin que la mayoritaria la ahogue. `scale_pos_weight = neg/pos` equivale a ponderar la clase minoritaria, práctica recomendada con razones extremas.

**4.3 Cómo debe leerse.**

Positivos y peso por etiqueta impresos por iteración (el peso de T1204 será de miles), lista de `skipped` y segundos totales (orden minutos en CPU por los 6 ajustes).

**4.4 Trazabilidad.**

Sus predicciones duras alimentan la evaluación de la sección 5. Complejidad: 6 × O(T · n · d) con T=300 árboles.



In [3]:
# 6 XGBoost binarios, uno por tecnica (scale_pos_weight = neg/pos)
import time
from xgboost import XGBClassifier

t0 = time.time()
P = np.zeros_like(Y[te])
S = np.zeros_like(Y[te], dtype=float)
skipped = []
for j, t in enumerate(TECHS):
    pos = int(Y[tr][:, j].sum()); neg = int(tr.sum()) - pos
    if pos < 2:
        skipped.append(t)
        continue
    clf = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08,
                          subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                          objective='binary:logistic', scale_pos_weight=neg / pos,
                          eval_metric='logloss', n_jobs=-1, random_state=42)
    clf.fit(Xtr, Y[tr][:, j])
    S[:, j] = clf.predict_proba(Xte)[:, 1]
    P[:, j] = (S[:, j] >= 0.5).astype(int)
    print(f'{t}: pos train {pos}  spw {neg/pos:.0f}', flush=True)
print(f'skipped (sin soporte train): {skipped} -> ceros')
print(f'total: {time.time()-t0:.0f}s')

T1190: pos train 91615  spw 0


T1041: pos train 4009  spw 22


T1567: pos train 4007  spw 22


T1204: pos train 10  spw 9161


skipped (sin soporte train): ['T1486', 'T1566'] -> ceros
total: 255s


#### 4.1 Observaciones e insights

**Observación.**

Seis ajustes completos (o menos por `skipped` en T1486/T1566), con pesos `neg/pos` por etiqueta y tiempo total en minutos CPU.

**Interpretación.**

El protocolo trata la cola con el máximo rigor barato disponible: ponderación exacta por desbalance. Si aun así las cabezas no se separan, no era falta de ponderación.

**Conexión.**

Anticipa el contraste de la sección 5: más capacidad y mejor ponderación contra el mismo techo.



### 5. Evaluación y contraste directo contra M4

**5.1 Qué se hace.**

Reporte por técnica, micro/macro-F1, hamming, subset-acc y PR-AUC por técnica, más impresión del JSON de M4 para contraste lado a lado; se persiste `models/m4b_metrics.json`.

**5.2 Por qué.**

El contraste impreso en la misma salida impide lecturas selectivas: la mejora global (micro) y el estancamiento estructural (macro) deben verse juntos. El PR-AUC se calcula sobre scores continuos `predict_proba`; las etiquetas sin positivos en test se reportan como 0.

**5.3 Cómo debe leerse.**

Comparar fila a fila contra M4: T1190 igual, T1041/T1567 peor en F1, T1204 mejor a costa de precisión, cola igual en cero. Global: micro sube por dominancia de T1190, macro no se mueve.

**5.4 Trazabilidad.**

Produce el artefacto citado en Limitations y el veredicto de despliegue de la sección 6.



In [4]:
from sklearn.metrics import (classification_report, f1_score, hamming_loss,
                             accuracy_score, average_precision_score)
print('=== M4b XGBoost por tecnica ===')
print(classification_report(Y[te], P, target_names=TECHS, zero_division=0))
res = {'model': 'm4b-xgb-perlabel-v1', 'labels': TECHS,
       'micro_f1': round(float(f1_score(Y[te], P, average='micro')), 4),
       'macro_f1': round(float(f1_score(Y[te], P, average='macro')), 4),
       'hamming': round(float(hamming_loss(Y[te], P)), 4),
       'subset_acc': round(float(accuracy_score(Y[te], P)), 4),
       'n_train': int(tr.sum()), 'n_test': int(te.sum())}
res['pr_auc'] = {t: (round(float(average_precision_score(Y[te][:, j], S[:, j])), 4) if int(Y[te][:, j].sum()) else 0.0) for j, t in enumerate(TECHS)}
print(res)
for j, t in enumerate(TECHS):
    ap = res['pr_auc'][t]
    print(f'{t}: soporte {int(Y[te][:, j].sum())}  PR-AUC(score) {ap}')
print('\nM4-LogReg:', json.loads((ROOT / 'models' / 'm4_metrics.json').read_text()))
(ROOT / 'models' / 'm4b_metrics.json').write_text(json.dumps(res, indent=2))
print('guardado models/m4b_metrics.json')

=== M4b XGBoost por tecnica ===


              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.06      0.07      3733
       T1567       0.08      0.06      0.07      3721
       T1204       0.24      1.00      0.39        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.82      0.77      0.79     30356
   macro avg       0.23      0.35      0.26     30356
weighted avg       0.77      0.77      0.77     30356
 samples avg       0.92      0.90      0.87     30356



{'model': 'm4b-xgb-perlabel-v1', 'labels': ['T1190', 'T1041', 'T1567', 'T1204', 'T1486', 'T1566'], 'micro_f1': 0.7926, 'macro_f1': 0.2557, 'hamming': 0.089, 'subset_acc': 0.7334, 'n_train': 91617, 'n_test': 22904, 'pr_auc': {'T1190': 0.9995, 'T1041': 0.1882, 'T1567': 0.1916, 'T1204': 0.3007, 'T1486': 0.0, 'T1566': 0.0}}
T1190: soporte 22892  PR-AUC(score) 0.9995
T1041: soporte 3733  PR-AUC(score) 0.1882
T1567: soporte 3721  PR-AUC(score) 0.1916
T1204: soporte 10  PR-AUC(score) 0.3007
T1486: soporte 0  PR-AUC(score) 0.0
T1566: soporte 0  PR-AUC(score) 0.0

M4-LogReg: {'model': 'm4-multilabel-v1', 'labels': ['T1190', 'T1041', 'T1567', 'T1204', 'T1486', 'T1566'], 'micro_f1': 0.7484, 'macro_f1': 0.2516, 'hamming': 0.1162, 'subset_acc': 0.6451, 'n_train': 91617, 'n_test': 22904, 'pr_auc': {'T1190': 1.0, 'T1041': 0.183, 'T1567': 0.1818, 'T1204': 0.2274, 'T1486': 0.0, 'T1566': 0.0}, 'oracle': {'micro_f1': 0.9994, 'macro_f1': 0.4997}}
guardado models/m4b_metrics.json


C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### 5.1 Observaciones e insights

**Observación.**

M4b: micro 0.79 (M4 0.75), macro 0.26 (M4 0.25), hamming 0.089, subset-acc 0.73. T1041/T1567: precisión 0.08, recall 0.06, F1 0.07 (peor que el lineal 0.10). T1204: precisión 0.24, recall 1.00, F1 0.39.

**Interpretación.**

H4c verificada: más capacidad empeora las cabezas (el XGB conservador ni dispara) y H4d también (la cola solo se caza con falsos positivos). La mejora del micro es espejismo de T1190. El techo es de la tarea a nivel evento, no del aprendiz.

**Conexión.**

Veredicto: NO desplegable como segunda cabeza (precisiones de 0.08 mentirían al analista). Queda como experimento y fila del paper. El camino real para la cola es otro: nivel de sentencia con MiniLM/SciBERT, aumento sintético estilo TTPXHunter con `incident_signals` y el mapeo tier→técnicas como capa honesta en `/score`.



### 6. Conclusión del cuaderno: el techo era de la tarea

#### 6.1 Resultados consolidados

| Contraste | Hallazgo | Valor |
|---|---|---|
| Global M4b vs M4 | micro sube, macro quieta | 0.79 vs 0.75; 0.26 vs 0.25 |
| Cabezas T1041/T1567 | más capacidad, peor F1 | 0.07 vs 0.10 del lineal |
| Cola T1204 | se caza con FPs | R 1.00, P 0.24, F1 0.39 |
| Cola T1486/T1566 | sin soporte | 0, skipped en train |
| Costo | 6 ajustes CPU | minutos, sin ganancia estructural |

#### 6.2 Cadena de hallazgos

- **El lineal no era el problema.** Con el mejor aprendiz no lineal del proyecto y ponderación exacta, las cabezas separan peor.

- **En live, las técnicas son priors de regla.** Derivan del tier por la correlación Precinct; a nivel evento no hay firma aprendible con estos features.

- **La mejora del micro miente si se reporta sola.** La domina T1190 constante; la macro quieta es el diagnóstico.

#### 6.3 Plan resultante

**Y** multilabel con X reutilizado y ponderación por etiqueta. Métricas en pareja micro/macro más subset-acc. PR-AUC sobre scores continuos, corregido en esta versión.

#### 6.4 Paso siguiente

`notebooks/06_m4c_multilabel_grafo.ipynb` (misma pregunta con contexto de grafo) y luego M4d por tácticas. Hints tier→técnicas en consola; sin modelo de atribución desplegado.

*Subir la capacidad y medir el mismo techo es lo que convierte la sospecha en evidencia.*

